In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
import pandas as pd
import numpy as np

# Load competition data (already mounted under /kaggle/input/)
train = pd.read_csv("/kaggle/input/house-prices-advanced-regression-techniques/train.csv")
test  = pd.read_csv("/kaggle/input/house-prices-advanced-regression-techniques/test.csv")

print(train.shape, test.shape)
train.head(3)

In [ ]:
# Handle missing values (simple fix)
train = train.fillna(train.median(numeric_only=True))
train = train.fillna("None")

test = test.fillna(test.median(numeric_only=True))
test = test.fillna("None")

# Confirm there are no missing values left
print("Remaining missing values in train:", train.isnull().sum().sum())
print("Remaining missing values in test:", test.isnull().sum().sum())

In [ ]:
from sklearn.model_selection import train_test_split

# Target and features (log1p target helps RMSE)
y = np.log1p(train["SalePrice"])
X = train.drop(columns=["SalePrice", "Id"])
X_test = test.drop(columns=["Id"])

# One-hot encode categoricals; align columns
X_enc      = pd.get_dummies(X)
X_test_enc = pd.get_dummies(X_test)
X_enc, X_test_enc = X_enc.align(X_test_enc, join="left", axis=1, fill_value=0)

# Train/valid split
X_tr, X_va, y_tr, y_va = train_test_split(X_enc, y, test_size=0.2, random_state=42)
X_enc.shape, X_test_enc.shape

In [ ]:
from sklearn.metrics import mean_squared_error
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
import numpy as np, pandas as pd

def rmse(y_true, y_pred):
    return np.sqrt(mean_squared_error(y_true, y_pred))

models = {
    "Linear Regression": LinearRegression(),
    "Random Forest": RandomForestRegressor(n_estimators=300, random_state=42, n_jobs=-1),
    "Gradient Boosting": GradientBoostingRegressor(random_state=42),
}

results = {}
for name, m in models.items():
    m.fit(X_tr, y_tr)
    preds = m.predict(X_va)
    results[name] = rmse(y_va, preds)

pd.DataFrame(results, index=["Valid_RMSE"]).T.sort_values("Valid_RMSE")

In [ ]:
# Pick the best by RMSE (fallback to Gradient Boosting if table is missing)
best_name = pd.Series(results).sort_values().index[0] if len(results) else "Gradient Boosting"
print("Best model:", best_name)

best_model = models[best_name]
best_model.fit(X_enc, y)

# Predict test (remember target was log1p → invert with expm1)
test_pred = np.expm1(best_model.predict(X_test_enc))

submission = pd.DataFrame({"Id": test["Id"], "SalePrice": test_pred})
submission.to_csv("/kaggle/working/submission.csv", index=False)
print("Saved:", "/kaggle/working/submission.csv")
submission.head()